# Teste isolado — Moody's Local Brasil (Ações de Rating)

Fonte **já existente** no catálogo (`nome_fonte="Moody's"`, setor
Regulatório/Múltiplo), hoje registrada com `source_id="site_page"`
(placeholder genérico compartilhado, herdado do bug de `source_id`
duplicado documentado em `CLAUDE.md`) e `status="Bloqueada"` — notas
antigas diziam "403 consistente mesmo com impersonation". Este notebook
**não** cria fonte nova nem duplicata; é só Fase 1 (descartável, sem
`atualizar_status_fonte`, sem gravar em `controle_fontes`).

Alvo desta rodada: a seção **"Últimas Ações de Rating"** de cada uma das
3 páginas de setor abaixo (mesma fonte "Moody's", não são 3 fontes
diferentes):

- https://moodyslocal.com.br/setores/projetos-de-infraestrutura/energia-infraestrutura/
- https://moodyslocal.com.br/setores/projetos-de-infraestrutura/transporte-infraestrutura/
- https://moodyslocal.com.br/setores/projetos-de-infraestrutura/outros-infraestrutura/

## Confirmado na exploração antes de escrever código

- **WordPress + Divi**, HTML renderizado no servidor — confirmado com
  `curl_cffi` (impersonation de browser), sem precisar de navegador.
- **403 é intermitente, não consistente** — diferente do que diziam as
  notas antigas em `controle_fontes`. Nas checagens desta Fase 1, cada
  uma das 3 URLs de setor devolveu 403 em pelo menos uma tentativa mas
  200 rodando de novo com outro perfil de impersonation (`transporte`
  chegou a levar 3 tentativas: `chrome120`/`chrome123` → 403,
  `chrome124` → 200). Mesma categoria de risco do bloqueio da ARTESP
  (WAF que alterna, não é 403/429 hard-block) — por isso o
  `baixar_pagina()` abaixo faz retry girando por vários perfis de
  impersonation + fallback `httpx`, igual ao padrão já usado no resto do
  projeto.
- Cada página de setor tem **duas seções de posts diferentes**, lado a
  lado no HTML (dois `.et_pb_posts` distintos) — fácil confundir:
  - `article.rating-action` (classe `type-rating-action`), precedida do
    `<h3>Últimas Ações de Rating</h3>` — **é essa a seção pedida**.
  - `article.issuer-report` (classe `type-issuer-report`, "Relatórios do
    Emissor") — seção **diferente**, fora de escopo aqui.

  O seletor usa a classe `rating-action` especificamente, então não
  captura a seção errada.
- Listagem: título+link em `h2.entry-title a`, data em
  `p.post-meta span.published`, formato `"Mon DD, YYYY"` (ex. `"Ago 17,
  2026"`) — meses abreviados em português (`Ago`, `Jul`) mas pelo menos
  um mês visto em inglês (`"Oct 30, 2024"`, num item fora da seção-alvo)
  — o dicionário de meses cobre as duas variantes por segurança.
- **Paginação é real** (confirmado comparando página 1 x página 2 da
  seção de Transporte: 7 itens cada, 0 sobreposição) — path
  `page/N/?et_blog`, diferente do caso do ABCR
  (`ingestores/TRANSPORTE/teste_abcr.ipynb`) onde `page/N/` não
  funcionava. Histórico gigantesco (Energia sozinha tem "Página 1 de
  91"), então usa `max_paginas=5` conservador — mesmo critério já usado
  em ABEGÁS/ABAR/Teletime.

## Página de detalhe (`/reporte/rating-action/...`) — resposta à pergunta da Fase 1

A página de detalhe **não tem o texto completo do rating no HTML** — o
corpo do post fica vazio (`<div class="et_pb_text_0_tb_body">` sem
conteúdo); o que existe é só título (`h1.entry-title`), data
(`.et_pb_title_meta_container .published`, formato **diferente** da
listagem: `"DD Mon YYYY"` sem vírgula, ex. `"18 Mar 2026"`) e um botão
**"Download"** (`<a class="et_pb_button" target="_blank"
href=".../algo.pdf">`) logo abaixo do título, que aponta pro PDF real do
relatório (`ML-BR-PR-<empresa>.pdf`). Confirmado em duas páginas de
detalhe diferentes — mesmo padrão nas duas. O botão "Download" é o
único link `target="_blank"` terminado em `.pdf` na página inteira (a
página também tem outros dois `.pdf` soltos no menu de navegação —
"Escalas de Rating", "Código de Conduta" — mas nenhum dos dois usa
`target="_blank"`, então não conflitam com o seletor).

O PDF baixado (testado com o primeiro item de Energia, 6 páginas) extrai
texto limpo via `pypdf` — 26.170 caracteres, sem OCR, sem página
escaneada.

**Conclusão da pergunta da Fase 1**: o texto completo **não** está na
página de detalhe — é preciso seguir o botão "Download" até o PDF de
verdade pra ter o conteúdo integral (mesmo padrão de 2 saltos já usado
em `listar_artemig_documentos()` no `ingest-PDF.ipynb`: listagem →
página de detalhe → PDF real).

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml pypdf
dbutils.library.restartPython()

In [0]:
import io
import re
import time
import random
import urllib.parse
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from pypdf import PdfReader

SETORES = {
    "energia": "https://moodyslocal.com.br/setores/projetos-de-infraestrutura/energia-infraestrutura/",
    "transporte": "https://moodyslocal.com.br/setores/projetos-de-infraestrutura/transporte-infraestrutura/",
    "outros": "https://moodyslocal.com.br/setores/projetos-de-infraestrutura/outros-infraestrutura/",
}

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

MESES = {
    "jan": 1, "fev": 2, "feb": 2, "mar": 3, "abr": 4, "apr": 4,
    "mai": 5, "may": 5, "jun": 6, "jul": 7, "ago": 8, "aug": 8,
    "set": 9, "sep": 9, "out": 10, "oct": 10, "nov": 11, "dez": 12, "dec": 12,
}

## Helper — download com retry (WAF intermitente)

Gira por vários perfis de `curl_cffi` impersonation antes de cair pra
`httpx` puro — necessário porque o 403 é intermitente (ver nota acima),
não um bloqueio consistente que precise de outra estratégia.

In [0]:
def baixar_pagina(url: str, tentativas: int = 4) -> Optional[str]:
    headers = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

    for tentativa in range(1, tentativas + 1):
        impersonate = IMPERSONATE_PROFILES[(tentativa - 1) % len(IMPERSONATE_PROFILES)]
        try:
            resp = cffi_requests.get(
                url, headers=headers, impersonate=impersonate,
                timeout=HTTP_TIMEOUT, allow_redirects=True,
            )
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi {impersonate} tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi {impersonate} tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    try:
        with httpx.Client(headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True) as client:
            resp = client.get(url)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx fallback] status={resp.status_code}")
    except Exception as e:
        print(f"  [httpx fallback] erro: {e}")

    return None


def parse_data(texto: str) -> Optional[str]:
    """Aceita 'Mon DD, YYYY' (listagem) ou 'DD Mon YYYY' (detalhe)."""
    if not texto:
        return None

    m = re.search(r"(\w{3,})\s+(\d{1,2}),\s*(\d{4})", texto)
    if m:
        mes_nome, dia, ano = m.groups()
    else:
        m = re.search(r"(\d{1,2})\s+(\w{3,})\s+(\d{4})", texto)
        if not m:
            return None
        dia, mes_nome, ano = m.groups()

    mes = MESES.get(mes_nome.lower()[:3])
    if not mes:
        return None
    return f"{ano}-{mes:02d}-{int(dia):02d}"

## Teste 1 — listar "Últimas Ações de Rating" das 3 páginas de setor

`article.rating-action` (classe específica, não pega a seção
"Relatórios do Emissor" que aparece na mesma página). Paginação real
confirmada — `max_paginas=5` conservador (histórico de até 91 páginas
só na Energia).

In [0]:
def listar_rating_actions_setor(nome_setor: str, url_setor: str, max_paginas: int = 5) -> list[dict]:
    itens = []

    for pagina in range(1, max_paginas + 1):
        url_pagina = url_setor if pagina == 1 else f"{url_setor.rstrip('/')}/page/{pagina}/?et_blog"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  [{nome_setor}] falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        artigos = soup.find_all("article", class_="rating-action")
        if not artigos:
            print(f"  [{nome_setor}] nenhum item em página {pagina}; fim da listagem.")
            break

        for artigo in artigos:
            tag_a = artigo.select_one("h2.entry-title a")
            if not tag_a:
                continue
            tag_data = artigo.select_one("p.post-meta span.published")

            itens.append({
                "titulo": tag_a.get_text(" ", strip=True),
                "url": urllib.parse.urljoin(url_setor, tag_a["href"].strip()),
                "published_at": parse_data(tag_data.get_text(strip=True) if tag_data else None),
                "setor_origem": nome_setor,
            })

        print(f"  [{nome_setor}] página {pagina}: {len(artigos)} itens.")
        time.sleep(random.uniform(0.6, 1.4))

    return itens


itens_por_setor = {}
for nome_setor, url_setor in SETORES.items():
    print(f"\n=== Setor: {nome_setor} ===")
    itens_por_setor[nome_setor] = listar_rating_actions_setor(nome_setor, url_setor, max_paginas=5)

for nome_setor, itens in itens_por_setor.items():
    print(f"\n{nome_setor}: {len(itens)} itens")
    for item in itens[:3]:
        print(f"  {item['published_at'] or '?':<12} {item['titulo'][:80]}")

## Teste 2 — página de detalhe: título, data e link do PDF real

Confirma o que já foi descrito na introdução: o texto completo não está
no HTML, só o botão "Download". Seletor: primeiro `a[target="_blank"]`
terminado em `.pdf` na página inteira (único candidato — os outros
`.pdf` do menu de navegação não têm `target="_blank"`).

In [0]:
def extrair_detalhe_rating_action(url: str) -> Optional[dict]:
    html = baixar_pagina(url)
    if not html:
        return None

    soup = BeautifulSoup(html, "lxml")

    h1 = soup.find("h1", class_="entry-title")
    titulo = h1.get_text(" ", strip=True) if h1 else None

    tag_data = soup.select_one(".et_pb_title_meta_container .published")
    published_at = parse_data(tag_data.get_text(strip=True) if tag_data else None)

    url_pdf = None
    for tag_a in soup.find_all("a", target="_blank", href=True):
        href = tag_a["href"].strip()
        if href.lower().endswith(".pdf"):
            url_pdf = urllib.parse.urljoin(url, href)
            break

    return {"titulo": titulo, "published_at": published_at, "url_pdf": url_pdf, "url_detalhe": url}


amostra_detalhes = []
todos_itens = [item for itens in itens_por_setor.values() for item in itens]
for item in todos_itens[:6]:
    detalhe = extrair_detalhe_rating_action(item["url"])
    if detalhe:
        amostra_detalhes.append(detalhe)
    print(f"  {'OK' if detalhe and detalhe['url_pdf'] else 'FALHA':<6} {item['titulo'][:70]}")
    time.sleep(random.uniform(0.6, 1.4))

print(f"\n{len(amostra_detalhes)}/{len(todos_itens[:6])} páginas de detalhe abertas com sucesso.")
sem_pdf = [d for d in amostra_detalhes if not d["url_pdf"]]
print(f"Sem PDF de download encontrado: {len(sem_pdf)}")
print(f"\nExemplo:")
print(amostra_detalhes[0])

## Teste 3 — baixar o PDF real e extrair o texto (`pypdf`)

Confirma que o relatório é PDF de texto de verdade, não página
escaneada.

In [0]:
def baixar_e_extrair_pdf(url_pdf: str) -> Optional[str]:
    headers = {"User-Agent": USER_AGENT}
    for impersonate in IMPERSONATE_PROFILES:
        try:
            resp = cffi_requests.get(url_pdf, headers=headers, impersonate=impersonate, timeout=HTTP_TIMEOUT)
            if resp.status_code == 200 and resp.content:
                reader = PdfReader(io.BytesIO(resp.content))
                return "\n".join(pagina.extract_text() or "" for pagina in reader.pages)
        except Exception as e:
            print(f"  [{impersonate}] erro: {e}")
    return None


detalhe_amostra = amostra_detalhes[0]
texto_pdf = baixar_e_extrair_pdf(detalhe_amostra["url_pdf"])

print(f"PDF: {detalhe_amostra['url_pdf']}")
print(f"Texto extraído: {len(texto_pdf) if texto_pdf else 0} chars\n")
print((texto_pdf or "")[:600])

## Teste 4 — dedup entre as 3 URLs de setor

Verifica se o mesmo rating action aparece em mais de uma página de
setor (esperado ser raro, já que cada relatório costuma pertencer a um
único setor, mas o dispatcher final precisa fazer esse dedup por URL do
item mesmo assim, mesmo padrão já usado nas outras fontes).

In [0]:
urls_por_setor = {nome: {item['url'] for item in itens} for nome, itens in itens_por_setor.items()}
total_bruto = sum(len(u) for u in urls_por_setor.values())

todas_urls = set()
for nome, urls in urls_por_setor.items():
    sobreposicao = todas_urls & urls
    if sobreposicao:
        print(f"  [{nome}] {len(sobreposicao)} item(ns) já visto(s) em outro setor.")
    todas_urls |= urls

print(f"\nTotal bruto (com possível repetição entre setores): {total_bruto}")
print(f"Total após dedup por URL do item: {len(todas_urls)}")

## Conclusão da Fase 1

`curl_cffi` com impersonation passa nas 3 páginas de setor e nas páginas
de detalhe — o 403 é **intermitente** (WAF que alterna, não bloqueio
consistente), resolvido com retry girando perfis de impersonation, sem
precisar de navegador real. As 3 URLs de setor pedidas compartilham a
mesma estrutura Divi (`article.rating-action`, precedido do `<h3>Últimas
Ações de Rating</h3>` — seção distinta de "Relatórios do Emissor" que
aparece na mesma página). Paginação real confirmada (`page/N/?et_blog`),
`max_paginas=5` conservador dado o histórico gigantesco (91 páginas só
em Energia).

A página de detalhe (`/reporte/rating-action/...`) **não** tem o texto
completo — só o botão "Download" que aponta pro PDF real do relatório
(`ML-BR-PR-<empresa>.pdf`), extraído sem problema via `pypdf` (26.170
caracteres num relatório de 6 páginas, sem OCR). Dedup por URL do item
funciona entre as 3 URLs de setor.

**Avaliação para a Fase 2**: não encaixa em `ingest-scraping` (o
conteúdo real está só no PDF, não no HTML) nem em `ingest-news-rss-infra`
(sem RSS). Encaixa em **`ingest-PDF`** — mesmo padrão de dois saltos já
usado em `listar_artemig_documentos()` (listagem → página de detalhe →
link do PDF real), com uma `listar_moodys()` própria que roda a mesma
lógica nas 3 URLs de setor e aplica o dedup por URL do item antes de
devolver a lista pro `processar_pdf()` genérico.

**Sobre o registro em `controle_fontes` (pendente pra Fase 3, não feito
neste notebook)**: a fonte já existe (`nome_fonte="Moody's"`) — a
integração deve fazer **UPDATE** na linha existente (`source_id` novo
único — ex. `moodys_local`, sem colidir com nada — `metodo_captura`,
`notebooks_responsaveis`, `status` para `"Coberta e validada em
produção"` depois de rodar em produção), **sem** criar linha nova nem
duplicata, e sem alterar `nome_fonte`. Nada foi gravado em
`controle_fontes` por este notebook — teste isolado, Fase 1 apenas.